In [46]:
# 네이버 리뷰 크롤링 날짜 필터(2026-02-13 이전 데이터만)
import re
import time
import pandas as pd
from datetime import datetime, date

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC


def parse_visit_date_from_time_text(raw: str):
    """
    raw 예시:
      - "1.5.월"
      - "25.12.23.화"
      - "2026.02.21."
    반환: "YYYY-MM-DD" 또는 None
    """
    if not raw:
        return None

    nums = re.findall(r"\d+", raw)

    # YYYY.MM.DD
    if len(nums) >= 3 and len(nums[0]) == 4:
        y, mo, d = int(nums[0]), int(nums[1]), int(nums[2])
        if 1 <= mo <= 12 and 1 <= d <= 31:
            return f"{y:04d}-{mo:02d}-{d:02d}"
        return None

    # YY.MM.DD
    if len(nums) >= 3:
        yy, mo, d = int(nums[0]), int(nums[1]), int(nums[2])
        y = 2000 + yy
        if 1 <= mo <= 12 and 1 <= d <= 31:
            return f"{y:04d}-{mo:02d}-{d:02d}"
        return None

    # MM.DD
    if len(nums) == 2:
        mo, d = int(nums[0]), int(nums[1])
        y = datetime.now().year
        if 1 <= mo <= 12 and 1 <= d <= 31:
            return f"{y:04d}-{mo:02d}-{d:02d}"
        return None

    return None


def ymd_to_date(ymd: str):
    if not ymd:
        return None
    y, m, d = ymd.split("-")
    return date(int(y), int(m), int(d))


def crawl_naver_reviews_cutoff(place_id, store_id, target_n=50, cutoff_ymd="2026-02-13", headless=False):
    cutoff_dt = ymd_to_date(cutoff_ymd)

    url = f"https://map.naver.com/p/entry/place/{place_id}"

    options = webdriver.ChromeOptions()
    if headless:
        options.add_argument("--headless=new")
    options.add_argument("--window-size=1200,900")
    options.add_argument("--lang=ko-KR")

    driver = webdriver.Chrome(options=options)
    wait = WebDriverWait(driver, 20)

    try:
        driver.get(url)

        # iframe 진입
        wait.until(EC.frame_to_be_available_and_switch_to_it((By.ID, "entryIframe")))

        # 리뷰 탭
        wait.until(EC.element_to_be_clickable((By.XPATH, "//a[normalize-space()='리뷰']"))).click()
        time.sleep(1)

        # 최신순
        wait.until(EC.element_to_be_clickable((By.XPATH, "//*[normalize-space()='최신순']"))).click()
        time.sleep(1.2)

        rows = []
        seen = set()

        # 더보기 버튼 XPath(네가 준 값)
        MORE_BTN_XPATH = "//*[@id='app-root']/div/div/div[7]/div[3]/div[3]/div[2]/div/a/span"

        # 안전장치: 더보기를 너무 많이 눌러도 못 채우면 중단
        max_more_clicks = 40
        more_clicks = 0

        while len(rows) < target_n:

            cards = driver.find_elements(By.XPATH, "//*[@id='_review_list']/li")

            for card in cards:
                if len(rows) >= target_n:
                    break

                text = card.text

                # reviewer_id (네가 준 XPath를 카드 기준 상대경로로)
                reviewer_id = None
                try:
                    reviewer_id = card.find_element(By.XPATH, ".//div[1]/a[2]/div[1]//span/span").text.strip()
                except:
                    try:
                        reviewer_id = card.find_element(By.XPATH, ".//div[1]//a[2]//span/span").text.strip()
                    except:
                        reviewer_id = None

                # date (time 태그)
                visit_date = None
                try:
                    raw_date = card.find_element(By.XPATH, ".//time").text.strip()
                    visit_date = parse_visit_date_from_time_text(raw_date)
                except:
                    visit_date = None

                # 날짜 파싱 실패면 제외(팀 기준상 날짜 없는 건 분석에 쓸모 없음)
                vd = ymd_to_date(visit_date) if visit_date else None
                if vd is None:
                    continue

                # ✅ 컷오프 적용: 2026-02-13 초과(=미래 데이터)는 스킵
                if vd > cutoff_dt:
                    continue

                # visit_count
                m = re.search(r"(\d+)\s*번째\s*방문", text)
                visit_count = int(m.group(1)) if m else None

                # visit_time
                m = re.search(r"(아침|점심|저녁|브런치|야간|새벽)", text)
                visit_time = m.group(1) if m else None

                # 중복 방지
                key = (reviewer_id, visit_date, visit_count, visit_time, text[:60])
                if key in seen:
                    continue
                seen.add(key)

                rows.append({
                    "store_id": store_id,
                    "reviewer_id": reviewer_id,
                    "visit_date": visit_date,
                    "visit_count": visit_count,
                    "visit_time": visit_time
                })

            if len(rows) >= target_n:
                break

            # 더보기 클릭해서 과거 리뷰 더 불러오기
            more_clicks += 1
            if more_clicks > max_more_clicks:
                break

            try:
                more_btn = driver.find_element(By.XPATH, MORE_BTN_XPATH)
                driver.execute_script("arguments[0].scrollIntoView({block:'center'});", more_btn)
                time.sleep(0.3)
                driver.execute_script("arguments[0].click();", more_btn)
                time.sleep(1.2)
            except:
                break

        df = pd.DataFrame(rows).head(target_n)
        out_path = f"../data/{store_id}_리뷰{target_n}개.xlsx"
        df.to_excel(out_path, index=False)
        return df, out_path

    finally:
        driver.quit()


if __name__ == "__main__":
    PLACE_ID = "91293725"
    STORE_ID = 1134

    df, path = crawl_naver_reviews_cutoff(
        PLACE_ID,
        STORE_ID,
        target_n=50,
        cutoff_ymd="2026-02-13",
        headless=False
    )
    print(df.head(10))
    print("Saved:", path)

   store_id reviewer_id  visit_date  visit_count visit_time
0      1134         장성욱  2026-02-09            2         저녁
1      1134  꿈은 이루어진다96  2026-02-09            1         저녁
2      1134      oa****  2026-02-07           18         아침
3      1134     jey****  2026-02-03            1         저녁
4      1134     모든것은타이밍  2026-02-02            1         점심
5      1134        최강히루  2026-01-30            9         점심
6      1134      호호맘539  2026-01-28            1         점심
7      1134      kwenkm  2026-01-27            1         저녁
8      1134       나란히88  2026-01-27           30         점심
9      1134       꿈꾸는37  2026-01-25            4         점심
Saved: ../data/1134_리뷰50개.xlsx


In [43]:
''' # 2026-02-13 날짜 필터 안함 버전
import re
import time
import pandas as pd
from datetime import datetime

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC


def parse_visit_date_from_time_text(raw: str):
    """
    raw 예시:
      - "1.5.월"
      - "25.12.23.화"
      - "2026.02.21."
    반환: "YYYY-MM-DD" 또는 None
    """
    if not raw:
        return None

    # 숫자 토큰만 추출 (요일 제거)
    nums = re.findall(r"\d+", raw)

    # 2026.02.21 같은 케이스: 3개 토큰이면서 첫 토큰이 4자리
    if len(nums) >= 3 and len(nums[0]) == 4:
        y, mo, d = int(nums[0]), int(nums[1]), int(nums[2])
        if 1 <= mo <= 12 and 1 <= d <= 31:
            return f"{y:04d}-{mo:02d}-{d:02d}"
        return None

    # 25.12.23 (YY.MM.DD)
    if len(nums) >= 3:
        yy, mo, d = int(nums[0]), int(nums[1]), int(nums[2])
        y = 2000 + yy  # 25 -> 2025
        if 1 <= mo <= 12 and 1 <= d <= 31:
            return f"{y:04d}-{mo:02d}-{d:02d}"
        return None

    # 1.5 (MM.DD)
    if len(nums) == 2:
        mo, d = int(nums[0]), int(nums[1])
        y = datetime.now().year
        if 1 <= mo <= 12 and 1 <= d <= 31:
            return f"{y:04d}-{mo:02d}-{d:02d}"
        return None

    return None


def crawl_naver_reviews(place_id, store_id, target_n=50, headless=False):

    url = f"https://map.naver.com/p/entry/place/{place_id}"

    options = webdriver.ChromeOptions()
    if headless:
        options.add_argument("--headless=new")
    options.add_argument("--window-size=1200,900")
    options.add_argument("--lang=ko-KR")

    driver = webdriver.Chrome(options=options)
    wait = WebDriverWait(driver, 20)

    try:
        driver.get(url)

        # iframe 진입
        wait.until(EC.frame_to_be_available_and_switch_to_it((By.ID, "entryIframe")))

        # 리뷰 탭
        wait.until(EC.element_to_be_clickable((By.XPATH, "//a[normalize-space()='리뷰']"))).click()
        time.sleep(1)

        # 최신순
        wait.until(EC.element_to_be_clickable((By.XPATH, "//*[normalize-space()='최신순']"))).click()
        time.sleep(1.2)

        rows = []
        seen = set()

        while len(rows) < target_n:

            cards = driver.find_elements(By.XPATH, "//*[@id='_review_list']/li")

            for card in cards:
                if len(rows) >= target_n:
                    break

                # --- reviewer_id (네가 준 XPath를 카드 기준 상대경로로) ---
                reviewer_id = None
                try:
                    reviewer_id = card.find_element(By.XPATH, ".//div[1]/a[2]/div[1]//span/span").text.strip()
                except:
                    # fallback: 다른 구조 대비
                    try:
                        reviewer_id = card.find_element(By.XPATH, ".//div[1]//a[2]//span/span").text.strip()
                    except:
                        reviewer_id = None

                # --- visit_date: time 태그에서 text 가져와서 파싱 ---
                visit_date = None
                try:
                    raw_date = card.find_element(By.XPATH, ".//time").text.strip()
                    visit_date = parse_visit_date_from_time_text(raw_date)
                except:
                    visit_date = None

                # --- visit_count / visit_time: 기존처럼 card.text에서 파싱 ---
                text = card.text

                m = re.search(r"(\d+)\s*번째\s*방문", text)
                visit_count = int(m.group(1)) if m else None

                m = re.search(r"(아침|점심|저녁|브런치|야간|새벽)", text)
                visit_time = m.group(1) if m else None

                key = (reviewer_id, visit_date, visit_count, visit_time, text[:60])
                if key in seen:
                    continue
                seen.add(key)

                rows.append({
                    "store_id": store_id,
                    "reviewer_id": reviewer_id,
                    "visit_date": visit_date,
                    "visit_count": visit_count,
                    "visit_time": visit_time
                })

            if len(rows) >= target_n:
                break

            # --- 더보기 클릭 (네가 준 XPath) ---
            try:
                more_btn = driver.find_element(By.XPATH,
                    "//*[@id='app-root']/div/div/div[7]/div[3]/div[3]/div[2]/div/a/span"
                )
                driver.execute_script("arguments[0].scrollIntoView({block:'center'});", more_btn)
                time.sleep(0.3)
                driver.execute_script("arguments[0].click();", more_btn)
                time.sleep(1.2)
            except:
                break

        df = pd.DataFrame(rows).head(target_n)
        out_path = f"naver_reviews_latest_{place_id}_{target_n}.xlsx"
        df.to_excel(out_path, index=False)
        return df, out_path

    finally:
        driver.quit()


if __name__ == "__main__":
    PLACE_ID = "36943347"
    STORE_ID = 947

    df, path = crawl_naver_reviews(PLACE_ID, STORE_ID, 50, headless=False)
    print(df.head(10))
    print("Saved:", path)
'''

<>:25: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<>:25: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
C:\Users\won24\AppData\Local\Temp\ipykernel_18408\1920145082.py:25: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
  nums = re.findall(r"\d+", raw)


' # 2026-02-13 날짜 필터 안함 버전\nimport re\nimport time\nimport pandas as pd\nfrom datetime import datetime\n\nfrom selenium import webdriver\nfrom selenium.webdriver.common.by import By\nfrom selenium.webdriver.support.ui import WebDriverWait\nfrom selenium.webdriver.support import expected_conditions as EC\n\n\ndef parse_visit_date_from_time_text(raw: str):\n    """\n    raw 예시:\n      - "1.5.월"\n      - "25.12.23.화"\n      - "2026.02.21."\n    반환: "YYYY-MM-DD" 또는 None\n    """\n    if not raw:\n        return None\n\n    # 숫자 토큰만 추출 (요일 제거)\n    nums = re.findall(r"\\d+", raw)\n\n    # 2026.02.21 같은 케이스: 3개 토큰이면서 첫 토큰이 4자리\n    if len(nums) >= 3 and len(nums[0]) == 4:\n        y, mo, d = int(nums[0]), int(nums[1]), int(nums[2])\n        if 1 <= mo <= 12 and 1 <= d <= 31:\n            return f"{y:04d}-{mo:02d}-{d:02d}"\n        return None\n\n    # 25.12.23 (YY.MM.DD)\n    if len(nums) >= 3:\n        yy, mo, d = int(nums[0]), int(nums[1]), int(nums[2])\n        y = 2000 + yy  # 25 -> 2025